# AutoSteer 2.0

## Getting started

### Introduction

The AutoSteer 2.0 architecture builds upon components of the YOLO11 framework, incorporating a specially designed head, and predicts the ego-path line directly from the input image. This ego-path representation captures the underlying road geometry and curvature, enabling more robust and accurate steering estimation.

AutoSteer 2.0 processess camera frames in a 2:1 aspect ratio with size 1024px by 512px and predicts ego-path line in camera coordinate system.

### Environment Setup



Install required dependencies

In [ ]:
%pip install --upgrade pip
%pip install -r ../../requirements.txt

# Verify the installation
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA availability: {torch.cuda.is_available()}")

### Download Models

Download the model weight files, using the links below. Subsequent tutorial sections assume models are downloaded inside directory:
`autoware_vision_pilot/Models/weights/`.

### [Link to Download Pytorch Model Weights *.pth](https://drive.google.com/file/d/1-wfPyu7HId7YDSh_T_DkB3Ma1zPc7Fu_/view?usp=drive_link)
### [Link to Download ONNX FP32 Weights *.onnx](https://drive.google.com/file/d/1Rb05CScAfvl0OPRJz8S83Q4BWRG1pEKH/view?usp=drive_link)

Download model files using a script bellow

In [ ]:
%pip install gdown
import os
import gdown

model_dir = "../../weights"
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

metadata = {
    "pth": {
        "url": "",
        "filepath": os.path.join(model_dir, "autosteer_2.0.pth")
    },
    "onnx": {
        "url": "",
        "filepath": os.path.join(model_dir, "autosteer_2.0.onnx")
    }
}

for weight_type in metadata:

    url = metadata[weight_type]["url"]
    filepath = metadata[weight_type]["filepath"]

    if not os.path.exists(filepath):
        print(f"Downloading AutoSpeed {weight_type} weights...")
        gdown.download(url, filepath, quiet=False)
    else:
        print(f"AutoSteer {weight_type} weights already exist at {filepath}. Skipping download.")

## Test Inference

### Video Inference

AutoSteer 2.0 video visualization is performed on video input. Input video is processed frame by frame, ego-path line is predicted in each frame, ego-path line points are drawn on the frame image and saved as a frame in the output video.

In [ ]:
MODEL_PATH = metadata["pth"]["filepath"]

INPUT_VIDEO = "../../tutorials/assets/videos/highway_1024_512.mp4"
OUTPUT_PATH = "../../tutorials/results/autosteer_2.0/videos/"
OUTPUT_VIDEO = os.path.join(OUTPUT_PATH, "highway_1024_512")

os.makedirs(OUTPUT_PATH, exist_ok=True)

# 2. Navigate to visualization directory and execute the script
!cd ../../visualizations/AutoSteer/ && \
python3 video_visualization.py \
    -a {os.path.abspath(MODEL_PATH)} \
    -i {os.path.abspath(INPUT_VIDEO)} \
    -o {os.path.abspath(OUTPUT_VIDEO)}

print(f"Saved video: {os.path.abspath(OUTPUT_VIDEO)}.avi")

Play the generated output video

In [ ]:
import os
from IPython.display import Video

# Display with specific dimensions and embedded data
print(f"Output video: {os.path.abspath(OUTPUT_VIDEO)}.avi")
Video(os.path.join(os.path.abspath(OUTPUT_VIDEO), "avi"), width=640, height=360, embed=True)

## Model Training

### Convert data

AutoSteer 2.0 model is trained on three datasets combined  [TUSimple](https://www.kaggle.com/datasets/manideep1108/tusimple), [OpenLane](https://github.com/OpenDriveLab/OpenLane) and [CurveLanes](https://github.com/SoulmateB/CurveLanes). In order to convert every dataset download the dataset into corresponding directory.

#### TUSimple

In [ ]:
import os

INPUT_DATASET_DIR = "<INPUT_DATASET_DIR>"
OUTPUT_DATASET_DIR = os.path.join(os.path.abspath(INPUT_DATASET_DIR), "_AutoSteer_2.0")

!cd ../../data_parsing/AutoSteer/TuSimple/ && \
python3 converter.py -i {os.path.abspath(INPUT_DATASET_DIR)} -o {OUTPUT_DATASET_DIR}

#### OpenLane

In [ ]:
import os

INPUT_DATASET_DIR = "<INPUT_DATASET_DIR>"
OUTPUT_DATASET_DIR = os.path.join(os.path.abspath(INPUT_DATASET_DIR), "_AutoSteer_2.0")

!cd ../../data_parsing/AutoSteer/OpenLane/ && \
python3 converter.py -i {os.path.abspath(INPUT_DATASET_DIR)} -o {OUTPUT_DATASET_DIR}

#### CurveLanes

In [ ]:
import os

INPUT_DATASET_DIR = "<INPUT_DATASET_DIR>"
OUTPUT_DATASET_DIR = os.path.join(os.path.abspath(INPUT_DATASET_DIR), "_AutoSteer_2.0")

!cd ../../data_parsing/AutoSteer/CurveLanes/ && \
python3 converter.py -i {os.path.abspath(INPUT_DATASET_DIR)} -o {OUTPUT_DATASET_DIR}

After all three datasets are converted combine all together in common directory example
```
AutoSteer_1.0
    ├── images/
    │     ├── train/
    │     └── val/
    └── labels/
          ├── train/
          └── val/
```

### Run training

In [ ]:
import os

DATASET_DIR = "AutoSteer_1.0"

!cd ../../training/ && \
python3 auto_steer_trainer.py --dataset {os.path.abspath(DATASET_DIR)}